# 03 — Feature engineering: perfiles disponibles antes del corte

## Pregunta que guía el notebook

Cómo convertir líneas de factura en un perfil por cliente sin mirar el futuro. Cada observación final representa a un cliente en una fecha de corte; la respuesta indica si compró durante los 30 días posteriores.

In [1]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
def find_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path('/content/retail_customer_intelligence')]:
        if (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError('No se encontró la raíz del proyecto. Abre la carpeta retail_customer_intelligence, no sólo el archivo .ipynb.')
ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src'))
from retail_demo import *
ensure_folders(ROOT)
print('Proyecto:', ROOT)

Proyecto: /Users/josemanuelromoperedo/Desktop/retail_customer_intelligence


## Convertir eventos en snapshots temporales

El perfil no usa un corte único. Se construye para varios meses: junio y julio entrenan, agosto permite elegir un umbral y septiembre permanece intacto para la prueba final. Así la evaluación se parece al uso real.

In [ ]:
raw = pd.read_parquet(ROOT / "data/interim/transacciones_originales.parquet")
clean = pd.read_parquet(ROOT / "data/processed/transacciones_limpias.parquet")
original, processed, manifest, target = build_model_tables(raw, clean)
leaky = make_leaky_table(processed, clean)

original.to_parquet(ROOT / "data/processed/modelo_original_minimo.parquet", index=False)
processed.to_parquet(ROOT / "data/processed/modelo_procesado.parquet", index=False)
leaky.to_parquet(ROOT / "data/processed/modelo_con_fuga_didactica.parquet", index=False)
manifest.to_parquet(ROOT / "data/processed/manifest_particion.parquet", index=False)

assert original[["CustomerID", "snapshot_date", "compra_futura"]].equals(
    processed[["CustomerID", "snapshot_date", "compra_futura"]]
)
temporal = processed.groupby(["snapshot_date", "particion"]).agg(
    clientes=("CustomerID", "size"),
    tasa_compra_futura=("compra_futura", "mean"),
    gasto_futuro_medio=("gasto_futuro", "mean"),
).reset_index()
display(temporal.round(3))

## La barrera temporal es más importante que el algoritmo

Cada predictor se calcula antes de la fecha de corte. La variable compra_futura sólo se construye con el mes siguiente. La línea temporal separa datos disponibles, validación y prueba; por eso una buena cifra posterior puede ser creíble.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.7))
ax.axvspan(pd.Timestamp("2011-06-01"), pd.Timestamp("2011-08-01"), color="#a8dadc", alpha=.8, label="entrenamiento")
ax.axvspan(pd.Timestamp("2011-08-01"), pd.Timestamp("2011-09-01"), color="#f4a261", alpha=.8, label="validación")
ax.axvspan(pd.Timestamp("2011-09-01"), pd.Timestamp("2011-10-01"), color="#e76f51", alpha=.8, label="prueba final")
for cutoff in SNAPSHOTS:
    ax.axvline(cutoff, color="#264653", linestyle="--")
ax.set(xlim=(pd.Timestamp("2011-06-01"), pd.Timestamp("2011-10-01")), yticks=[], title="Orden temporal de entrenamiento, validación y prueba")
ax.legend(ncol=3, loc="upper center")
plt.tight_layout()

## Variables con significado, no identificadores sin contexto

Las familias de variables resumen actividad reciente, frecuencia, valor, diversidad, tendencia y devoluciones. No se trata de inventar columnas: se trata de expresar lo que el historial ya dice antes del corte.

In [ ]:
dictionary = pd.DataFrame([
    ("recencia_dias", "Días desde última venta válida", "actividad reciente"),
    ("frecuencia_facturas", "Facturas válidas históricas", "frecuencia"),
    ("gasto_historico", "Importe válido acumulado", "valor"),
    ("ticket_promedio y ticket_mediano", "Tamaño habitual de una factura", "valor"),
    ("productos_distintos", "Variedad de productos identificables", "diversidad"),
    ("gasto_30d y gasto_90d", "Actividad en ventanas móviles", "recencia e intensidad"),
    ("tendencia_gasto_30d", "Mes reciente contra mes previo", "tendencia"),
    ("tasa_devolucion", "Devoluciones separadas de ventas", "calidad de relación"),
], columns=["variable", "significado", "familia"])
dictionary.to_csv(ROOT / "reports/diccionario_variables.csv", index=False)
display(dictionary)

sample = processed.sample(min(2500, len(processed)), random_state=SEED)
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=sample, x="recencia_dias", y="log_gasto_historico", hue="compra_futura", alpha=.55, ax=ax)
ax.set(title="Señales construidas antes de conocer el objetivo")
plt.tight_layout()

## Auditoría de fuga: un resultado brillante que debemos rechazar

Se crea un dataset didáctico con la columna lineas_posteriores_al_corte. Esa columna cuenta actividad del mismo periodo que queremos predecir. Se usará una sola vez para mostrar por qué una métrica alta puede ser evidencia de un pipeline incorrecto.

In [ ]:
availability = pd.DataFrame([
    ("modelo_original_minimo", "agregados crudos antes del corte", "válido, pero limitado"),
    ("modelo_procesado", "perfil limpio antes del corte", "válido"),
    ("modelo_con_fuga_didactica", "actividad posterior al corte", "INVÁLIDO; sólo demostración"),
], columns=["dataset", "información", "veredicto"])
display(availability)
assert "lineas_posteriores_al_corte" not in processed.columns
assert "lineas_posteriores_al_corte" in leaky.columns
display(processed.isna().mean().sort_values(ascending=False).head(12).rename("proporción de nulos").to_frame().round(3))

## Mensaje para clase

Feature engineering es una decisión de representación: convierte eventos en señales que existirían cuando se decide a quién contactar. La pregunta temporal protege contra la fuga de información.

Artefactos: datasets de modelado, manifiesto temporal y reports/diccionario_variables.csv.